In [2]:
# Parallel Chains

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
import os

c:\Users\arunk\anaconda3\envs\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

In [5]:
prompt1 = PromptTemplate(template='Generate a summary on this {Topic}', input_variables=['Topic'])

In [6]:
prompt2 = PromptTemplate(template='Generate 5 quiz question on this {Topic}', input_variables=['Topic'])

In [7]:
gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [8]:
gpt = ChatOpenAI(model='gpt-3.5-turbo')

In [9]:
groq = ChatGroq(model='llama-3.3-70b-versatile', api_key= os.getenv("GROQ_API_KEY"))

In [10]:
parser = StrOutputParser()

In [11]:
parallel_chain=  RunnableParallel({
    'summary': prompt1 | groq | parser,
    'quiz' : prompt2 | gpt | parser
})

In [12]:
prompt3 = PromptTemplate(template='Merge the following {summary} and {quiz} together', input_variables=['summary', 'quiz'])

In [13]:
chain3 = prompt3 | groq | parser

In [14]:
final_chain = parallel_chain | chain3

In [15]:
result = final_chain.invoke({'Topic':'Generative AI'})

In [16]:
result

"**Introduction to Generative AI**\n\nGenerative AI refers to a type of artificial intelligence that is capable of generating new, original content, such as images, videos, music, text, and more. This technology uses complex algorithms and neural networks to learn patterns and relationships within existing data, allowing it to create novel and coherent outputs. To understand Generative AI, it's essential to address some key questions: \n\n1. **What is Generative AI and how does it differ from other types of artificial intelligence?** \nGenerative AI is a subset of artificial intelligence that focuses on generating new content, whereas other types of AI might focus on analyzing or processing existing data. \n\n2. **How does Generative AI work, specifically in terms of creating new content?** \nGenerative AI works by using complex algorithms and neural networks to learn patterns and relationships within existing data. This learning process enables the AI to generate new content that is s

In [17]:
final_chain.get_graph().print_ascii()

          +-----------------------------+          
          | Parallel<summary,quiz>Input |          
          +-----------------------------+          
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +------------+   
    | ChatGroq |                  | ChatOpenAI |   
    +----------+                  +------------+   
          *                             *          
          *                             *          
          *                             *          
+-----------------+            +-----------------+ 
| StrOutputP